In [1]:
# Load and save regional mortality in single files

In [2]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config

In [3]:
# Number of samples: full distribution or stats
n_samples = 300

In [11]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}-{years.stop - 1}"

MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/region/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/"

for ens_num in ensemble_members:
    print(f"Processing ensemble number {ens_num:02d}")

    ds_years = []
    for year in years:
        if n_samples <= 500:
            files = f"Regional_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_*_{year}.nc"
            description = ("Regional mortality (COPD) due to ozone "
                           " - scripts by A.F. Wells (2025)")
            out_file = f"Regional_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"

        else:
            files = f"Regional_mortality_stats_{model}_{scenario}_{ens_num:02d}_*_{year}.nc"
            description = ("Regional mortality (COPD) due to ozone "
                           "statistics: including mean, median, "
                           "and the 95% CI - scripts by A.F. Wells (2025)")
            out_file = f"Regional_mortality_stats_{model}_{scenario}_{ens_num:02d}_{dates}.nc"

        file_path = os.path.join(MORT_DIR, files)

        ds = xr.open_mfdataset(
            sorted(glob.glob(file_path)),
            combine="nested",
            concat_dim="region")
        ds_years.append(ds)

    ds_mort = xr.concat(
        ds_years,
        dim=xr.DataArray(
            years,
            dims="year",
            name="year")
    )

    ds_mort.attrs["description"] = description
    ds_mort.attrs["model"] = model
    ds_mort.attrs["scenario"] = scenario
    ds_mort.attrs["ensemble_number"] = ens_num

    out_path = os.path.join(SAVE_DIR, out_file)
    ds_mort.to_netcdf(out_path, engine="h5netcdf", encoding={"region": {"dtype": str}})

print("All processing complete.")

Processing ensemble number 01


ValueError: coordinate 'region' not present in all datasets.

In [25]:
da_list = []
for i in glob.glob(file_path):
    da = xr.open_dataarray(i)
    da_list.append(da)

In [34]:
new_da_list = []
for da in da_list:
    region = da.attrs["region"]
    da = da.expand_dims({"region": [region]})
    new_da_list.append(da)

In [35]:
new_da_list

[<xarray.DataArray (region: 1, samples: 300)> Size: 2kB
 array([[1.58373968e+08, 6.05022372e+08, 1.77422808e+08, 2.13144322e+08,
         5.47832096e+08, 6.73264013e+08, 7.71546014e+08, 8.54527518e+08,
         7.85803708e+08, 7.81172561e+08, 7.91456341e+08, 5.12855025e+08,
         1.74921822e+08, 1.90731452e+08, 6.00392292e+08, 6.55029593e+08,
         7.68823777e+08, 5.32828067e+08, 8.25641637e+08, 6.65739356e+08,
         5.53033154e+08, 3.74892979e+08, 5.73566458e+08, 7.67896844e+08,
         7.34672050e+08, 4.45018659e+08, 5.13033349e+08, 3.70355481e+08,
         1.70686008e+08, 7.75810479e+08, 2.26802636e+08, 4.57747438e+08,
         1.68010959e+08, 7.30981924e+08, 6.22425918e+08, 5.21150381e+08,
         7.40560787e+08, 6.86240335e+08, 6.94472778e+08, 3.25350456e+08,
         8.08714156e+08, 8.02706048e+08, 6.73828653e+08, 4.78297722e+08,
         7.44257563e+08, 6.46338912e+08, 4.61345706e+08, 3.01367918e+08,
         7.67023616e+08, 7.01922880e+08, 7.30003534e+08, 7.61907473e

In [36]:
combined = xr.concat(new_da_list, dim="region")

In [39]:
combined

<xarray.DataArray (region: 21, samples: 300)> Size: 50kB
array([[1.58373968e+08, 6.05022372e+08, 1.77422808e+08, ...,
        1.84942718e+08, 6.94229735e+08, 6.95227960e+08],
       [6.92849209e+06, 0.00000000e+00, 9.39480601e+06, ...,
        6.27994343e+06, 0.00000000e+00, 0.00000000e+00],
       [7.23601525e+08, 6.22350930e+07, 8.93537570e+08, ...,
        5.33639924e+08, 1.18583158e+08, 6.95828974e+07],
       ...,
       [1.76001133e+08, 2.54933652e+08, 1.75227563e+08, ...,
        2.19007440e+08, 2.65273620e+08, 2.80472814e+08],
       [1.20458545e+08, 0.00000000e+00, 9.83066930e+07, ...,
        1.36123730e+08, 7.70386560e+05, 0.00000000e+00],
       [1.66374147e+08, 0.00000000e+00, 1.62426781e+08, ...,
        1.34581381e+08, 3.13536633e+07, 0.00000000e+00]])
Coordinates:
  * region   (region) object 168B 'Southern Latin America' ... 'High-income A...
  * samples  (samples) int64 2kB 0 1 2 3 4 5 6 7 ... 293 294 295 296 297 298 299
    year     int64 8B 2020
Attributes:
    description:      Regional mortality (COPD) due to ozone sample size 300 ...
    model:            CESM2
    scenario:         SSP245_G6
    ensemble_number:  1
    region:           Southern Latin America
    year:             2020